# Attention sink visualization with 4 or 8 bit quantization for Llama in BertViz
Make sure that you can access the Llama models from huggingface:

https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

https://huggingface.co/meta-llama/Meta-Llama-3-8B

Afterwards copy the access token from the huggingface setting for the login.

In [ ]:
pip install bitsandbytes accelerate transformers bertviz --quiet

In [ ]:
!huggingface-cli login

In [ ]:
from transformers import AutoTokenizer, AutoModel, utils
#utils.logging.set_verbosity_error()  # Suppress standard warnings

# load the model
# meta-llama/Meta-Llama-3-8B
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# load the model in 4 bit
# The VRAM of google colab also fits 8 bit
model = AutoModel.from_pretrained(model_name, output_attentions=True, load_in_4bit=True, device_map="auto")

In [ ]:
input_sentence = """The sentence you are given might be too wordy, complicated, or unclear. Rewrite the sentence and make your writing clearer by keeping it concise. Whenever possible, break complex sentences into multiple sentences and eliminate unnecessary words.\n
Input: If you have any questions about my rate or if you find it necessary to increase or decrease the scope for this project, please let me know.\n\nOutput:"""
input_sentence = "Generate a positive review for a place."
inputs = tokenizer.encode(input_sentence, return_tensors='pt')
outputs = model(inputs)
attention = outputs[-1]  # Output includes attention weights when output_attentions=True
tokens = tokenizer.convert_ids_to_tokens(inputs[0])
# save the attention weights into a file
import pickle
with open('attention.pkl', 'wb') as f:
    pickle.dump(attention, f)
    
# save the tokens into a file
with open('tokens.pkl', 'wb') as f:
    pickle.dump(tokens, f)
    


In [ ]:
import pickle
from bertviz import head_view
from bertviz import model_view
# load the attention weights from the file
with open('../data/tensor/attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open('../data/tensor/tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")
html_head_view = head_view(attention, tokens)

In [ ]:
tokens

In [ ]:
model_view(attention, tokens)

# Further literature

A Primer on the Inner Workings of Transformer-based Language Models: https://arxiv.org/abs/2405.00208

Massive Activations in Large Language Models
https://arxiv.org/abs/2402.17762

Understanding and Minimising Outlier Features in Neural Network Training
https://arxiv.org/abs/2405.19279

signals exchanged in the tail end of the spectrum are responsible for
attention sinking
https://arxiv.org/abs/2402.09221

Information Flow Routes: Automatically Interpreting Language Models at Scale
https://arxiv.org/abs/2403.00824

MInference 1.0: Accelerating Pre-filling for Long-Context LLMs via Dynamic Sparse Attention https://huggingface.co/papers/2407.02490

Efficient implementations of state-of-the-art linear attention models in Pytorch and Triton https://github.com/sustcsonglin/flash-linear-attention

In [ ]:
import pickle
from bertviz import head_view
from bertviz import model_view
# load the attention weights from the file
with open('../data/tensor/attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open('../data/tensor/tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")

In [ ]:

# save the complete model view
html_head_view = head_view(attention, tokens, html_action='return')
with open("all_head_view.html", 'w') as file:
    file.write(html_head_view.data)

html_model_view = model_view(attention, tokens, html_action='return')
with open("all_model_view.html", 'w') as file:
    file.write(html_model_view.data)

# save the view just for certain layers if the browser can't display the whole
# shorter inputs are easier to display
layers = [1]
html_head_view = head_view(attention, tokens, html_action='return', include_layers=layers)

with open("short_head_view.html", 'w') as file:
    file.write(html_head_view.data)

html_model_view = model_view(attention, tokens, html_action='return', include_layers=layers)
with open("short_model_view.html", 'w') as file:
    file.write(html_model_view.data)

In [ ]:
html_head_view

In [ ]:
html_model_view

In [ ]:
attention[0]

In [ ]:
attention[0].shape

In [ ]:
attention[0][0][0][110]

In [35]:
import torch
#3-15 grd for title
# -10: for question
topk = 30
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 15 and val >= 3:
                count += 1
    print(f"Layer: {layer} Count: {count}")

Layer: 0 Count: 8
Layer: 1 Count: 0
Layer: 2 Count: 20
Layer: 3 Count: 23
Layer: 4 Count: 10
Layer: 5 Count: 6
Layer: 6 Count: 10
Layer: 7 Count: 12
Layer: 8 Count: 18
Layer: 9 Count: 16
Layer: 10 Count: 29
Layer: 11 Count: 15
Layer: 12 Count: 5
Layer: 13 Count: 25
Layer: 14 Count: 29
Layer: 15 Count: 28
Layer: 16 Count: 28
Layer: 17 Count: 36
Layer: 18 Count: 33
Layer: 19 Count: 36
Layer: 20 Count: 36
Layer: 21 Count: 40
Layer: 22 Count: 38
Layer: 23 Count: 37
Layer: 24 Count: 37
Layer: 25 Count: 24
Layer: 26 Count: 13
Layer: 27 Count: 21
Layer: 28 Count: 19
Layer: 29 Count: 24
Layer: 30 Count: 31


In [37]:
import pickle
from bertviz import head_view
from bertviz import model_view
# load the attention weights from the file
with open('../data/tensor/abstract_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open('../data/tensor/abstract_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")

Attention shape: 32
tokens length: 112


In [38]:
import torch
#71-99 grd for abstract
# -10: for question
topk = 30
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 99 and val >= 71:
                count += 1
    print(f"Layer: {layer} Count: {count}")

Layer: 0 Count: 165
Layer: 1 Count: 154
Layer: 2 Count: 103
Layer: 3 Count: 139
Layer: 4 Count: 144
Layer: 5 Count: 133
Layer: 6 Count: 142
Layer: 7 Count: 119
Layer: 8 Count: 104
Layer: 9 Count: 103
Layer: 10 Count: 101
Layer: 11 Count: 102
Layer: 12 Count: 121
Layer: 13 Count: 88
Layer: 14 Count: 85
Layer: 15 Count: 88
Layer: 16 Count: 87
Layer: 17 Count: 91
Layer: 18 Count: 100
Layer: 19 Count: 92
Layer: 20 Count: 84
Layer: 21 Count: 91
Layer: 22 Count: 94
Layer: 23 Count: 88
Layer: 24 Count: 91
Layer: 25 Count: 99
Layer: 26 Count: 103
Layer: 27 Count: 92
Layer: 28 Count: 107
Layer: 29 Count: 103
Layer: 30 Count: 93


In [ ]:
# return the top 10 values's indices
topk = torch.topk(sum[-10:], 20)
topk.indices


In [ ]:
tokens[71:104]

In [36]:
tokens

['<|begin_of_text|>',
 'Context',
 ':',
 'ĠA',
 'ĠDeep',
 'ĠDive',
 'Ġinto',
 'ĠCommon',
 'ĠOpen',
 'ĠFormats',
 'Ġfor',
 'ĠAnaly',
 'tical',
 'ĠDB',
 'MS',
 's',
 'Ċ',
 'Ch',
 'un',
 'wei',
 'ĠLiu',
 'ĠMIT',
 'ĠCS',
 'AIL',
 'Ġch',
 'un',
 'wei',
 '@',
 'cs',
 'ail',
 '.mit',
 '.edu',
 'Ċ',
 'AB',
 'STRACT',
 'Ċ',
 'Anna',
 'ĠPav',
 'len',
 'ko',
 'ĠMicrosoft',
 'Ġann',
 'apa',
 '@m',
 'icrosoft',
 '.com',
 'Ċ',
 'Mat',
 'te',
 'o',
 'ĠInter',
 'land',
 'i',
 'ĠMicrosoft',
 'Ġmaint',
 'er',
 'l',
 '@m',
 'icrosoft',
 '.com',
 'Ċ',
 'Brandon',
 'ĠHay',
 'nes',
 'ĠMicrosoft',
 'Ġbr',
 'hay',
 'nes',
 '@m',
 'icrosoft',
 '.com',
 'Ċ',
 'This',
 'Ġpaper',
 'Ġevaluates',
 'Ġthe',
 'Ġsuitability',
 'Ġof',
 'ĠApache',
 'ĠArrow',
 ',',
 'ĠPar',
 'quet',
 ',',
 'Ġand',
 'ĠOR',
 'C',
 'Ġas',
 'Ġformats',
 'Ġfor',
 'Ġsub',
 'sum',
 'ption',
 'Ġin',
 'Ġan',
 'Ġanalytical',
 'ĠDB',
 'MS',
 '.',
 'ĠĊĊ',
 'Question',
 ':',
 'ĠWhat',
 'Ġis',
 'Ġthe',
 'Ġtitle',
 'Ġof',
 'Ġthe',
 'Ġpaper',
 '?Ċ',
 'An